# Notebook 3 — Production RAG
## PostgreSQL + pgvector, HNSW, evaluation, and observability

Notebooks 1 and 2 built retrieval that *works*. This notebook makes it **operable** — the difference
between a demo and something you'd page someone about at 3am.

Three things change:

| | Notebooks 1–2 | Notebook 3 |
|---|---|---|
| **Storage** | in-memory Python list / FAISS | PostgreSQL + pgvector (durable, transactional, filterable) |
| **Index** | exact brute force | HNSW — and we *measure* the recall it costs us |
| **Quality** | `hit@1` on 10 questions | Precision@k, Recall@k, MRR, nDCG — implemented from scratch |
| **Ops** | none | latency percentiles, score-drift detection, structured query logs |

**On RAGAS.** The original plan named RAGAS for evaluation. I'm deliberately *not* using it here, for
two reasons: its core metrics call an LLM to judge answers, which breaks this series' no-API-key
property; and it drags in the LangChain dependency chain we agreed to defer until Notebook 4.
More importantly — RAGAS's *retrieval* metrics are ordinary IR arithmetic. Implementing them yourself
in ~20 lines is how you learn what the library reports. We'll wire up RAGAS in Notebook 4, once you
know what its numbers mean.

> ⚠️ **Runtime:** CPU is fine. Section 1 installs PostgreSQL inside the Colab VM (~2 min).
> Colab VMs are ephemeral — the database dies with the runtime. That's fine; it's a teaching database.


---
## Section 0 — Why a real database at all?

FAISS was fast and it worked. So why move to Postgres?

FAISS is an **index**, not a **database**. It holds vectors and returns neighbors. It does not give you:

- **Durability** — FAISS lives in RAM. Process restarts, index gone. Re-embedding a large corpus costs real money.
- **Transactions** — add a runbook and its metadata atomically, or roll both back.
- **Rich filtering** — `WHERE service = 'payment' AND updated_at > now() - interval '90 days'`.
  In Notebook 2 we hand-rolled this by slicing numpy arrays. SQL does it properly, with indexes.
- **One source of truth** — chunk text, embedding, metadata, and access controls in the same row.
  No sync problem between "the vector store" and "the metadata store".
- **Operational maturity** — backups, replication, monitoring, and a DBA skill set your org already has.

**The trade-off, stated honestly:** a dedicated vector DB (Milvus, Qdrant, Pinecone) will beat pgvector
on raw throughput at very high scale. pgvector's pitch is that it's *already in your stack*. For the
SRE Copilot — where incidents are thousands, not billions — "one less system to operate during an
outage" is worth far more than peak QPS.


---
## Section 1 — Get PostgreSQL + pgvector running

**Two paths. Pick one.**

### Path A — your own Ubuntu machine, via Docker (recommended)

This is the better option and it's the same container you'll use in Milestone 13, so the work carries
forward. Run this on `aak-hp`, not in the notebook:

```bash
mkdir -p ~/learning/sre-copilot/docker/pgvector && cd ~/learning/sre-copilot/docker/pgvector

cat > docker-compose.yml <<'YAML'
services:
  pgvector:
    image: pgvector/pgvector:pg16
    container_name: sre-pgvector
    environment:
      POSTGRES_USER: postgres
      POSTGRES_PASSWORD: postgres
      POSTGRES_DB: srecopilot
    ports:
      - "5432:5432"
    volumes:
      - pgdata:/var/lib/postgresql/data
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U postgres -d srecopilot"]
      interval: 5s
      retries: 10
volumes:
  pgdata:
YAML

docker compose up -d
docker compose exec pgvector psql -U postgres -d srecopilot -c "CREATE EXTENSION IF NOT EXISTS vector;"
docker compose exec pgvector psql -U postgres -d srecopilot -c "SELECT extversion FROM pg_extension WHERE extname='vector';"
```

Three things worth noticing in that compose file, because they're Milestone 2 concepts in the wild:
the **named volume** (`pgdata`) is what makes your data survive `docker compose down` — without it
you'd re-embed the corpus every restart; the **healthcheck** is what lets dependent services wait for
readiness rather than crash-looping; and the **port mapping** exposes 5432 on the host so the notebook
can connect. Then run this notebook locally with `jupyter lab`, or point Colab at your machine.

### Path B — inside Colab (ephemeral, zero setup)

Run the cell below. It installs Postgres in the Colab VM and builds pgvector from source (~3 min).
The database dies with the runtime — fine for learning, useless for anything else.

> **Set `USE_DOCKER = True` in the config cell if you're on Path A.**


In [ ]:
# ===== PATH B ONLY — skip this cell entirely if you're using Docker locally =====
USE_DOCKER = False   # ← set True if Postgres is already running (Path A)

if not USE_DOCKER:
    import subprocess
    print("Installing PostgreSQL + pgvector inside Colab (~3 min)...")
    subprocess.run('''
      sudo apt-get -qq update > /dev/null 2>&1
      sudo apt-get -y -qq install postgresql postgresql-server-dev-16 build-essential git > /dev/null 2>&1
      if [ ! -d /tmp/pgvector ]; then
        git clone --branch v0.7.4 --depth 1 https://github.com/pgvector/pgvector.git /tmp/pgvector > /dev/null 2>&1
      fi
      cd /tmp/pgvector && make > /dev/null 2>&1 && sudo make install > /dev/null 2>&1
      sudo service postgresql start
      sleep 4
      sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'postgres';"
      sudo -u postgres psql -c "DROP DATABASE IF EXISTS srecopilot;"
      sudo -u postgres psql -c "CREATE DATABASE srecopilot;"
      sudo -u postgres psql -d srecopilot -c "CREATE EXTENSION IF NOT EXISTS vector;"
    ''', shell=True, executable="/bin/bash")
    print("Done.")
else:
    print("Skipping Colab install — expecting Postgres at localhost:5432 (Docker).")

In [ ]:
%pip install -q sentence-transformers psycopg2-binary pgvector rank_bm25

In [ ]:
import re, time, json, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import psycopg2
from psycopg2.extras import execute_values, RealDictCursor
from pgvector.psycopg2 import register_vector
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_colwidth", 60)
plt.rcParams["figure.figsize"] = (10, 5)

DSN = "host=localhost dbname=srecopilot user=postgres password=postgres"
conn = psycopg2.connect(DSN)
conn.autocommit = True
register_vector(conn)          # lets psycopg2 send/receive numpy arrays as vector type

with conn.cursor() as cur:
    cur.execute("SELECT 1")
    print("Postgres connection OK ✅", cur.fetchone())

---
## Section 2 — Schema design

The schema *is* an architecture decision. Points worth defending in a design review:

- **`embedding vector(384)`** — dimension is fixed at DDL time. Changing embedding models means a
  migration, not a config flag. (This is Notebook 1 quiz #2 made physical: the column literally
  cannot hold vectors from a different model.)
- **`embedding_model` column** — stamp every row with the model that produced it. When you migrate,
  you can find stale rows with a `WHERE` clause instead of guessing. Skipping this is how the
  mismatched-model bug becomes undebuggable in production.
- **Metadata as real columns** (`service`, `severity`) rather than a JSON blob — they're filter keys,
  so they need B-tree indexes.
- **`content_hash`** — deduplication. Re-ingesting an unchanged runbook shouldn't re-embed it.
  Embedding calls cost money and time; this is the cheapest optimization in the whole pipeline.
- **`updated_at`** — enables recency filtering and staleness alerts. A runbook nobody has touched in
  two years is a liability during an incident.


In [ ]:
SCHEMA = """
DROP TABLE IF EXISTS runbook_chunks;

CREATE TABLE runbook_chunks (
    id              BIGSERIAL PRIMARY KEY,
    doc             TEXT        NOT NULL,          -- source runbook
    chunk_id        TEXT        NOT NULL UNIQUE,   -- e.g. 'kafka-cluster#2'
    chunk_index     INT         NOT NULL,
    content         TEXT        NOT NULL,
    content_hash    TEXT        NOT NULL,          -- dedup / change detection
    service         TEXT,                          -- metadata filter key
    severity        TEXT,                          -- metadata filter key
    embedding       vector(384) NOT NULL,          -- dimension fixed at DDL time!
    embedding_model TEXT        NOT NULL,          -- which model produced this vector
    updated_at      TIMESTAMPTZ NOT NULL DEFAULT now()
);

CREATE INDEX idx_chunks_service  ON runbook_chunks (service);
CREATE INDEX idx_chunks_severity ON runbook_chunks (severity);
CREATE INDEX idx_chunks_updated  ON runbook_chunks (updated_at DESC);
CREATE INDEX idx_chunks_fts      ON runbook_chunks USING GIN (to_tsvector('english', content));
"""

with conn.cursor() as cur:
    cur.execute(SCHEMA)
print("Schema created ✅")

with conn.cursor(cursor_factory=RealDictCursor) as cur:
    cur.execute("""SELECT column_name, data_type FROM information_schema.columns
                   WHERE table_name='runbook_chunks' ORDER BY ordinal_position;""")
    display(pd.DataFrame(cur.fetchall()))

**Note that last index — `GIN (to_tsvector(...))`.** That's Postgres full-text search. It means
we can do the *hybrid* search from Notebook 2 entirely inside the database: BM25-style keyword ranking
via `ts_rank`, vector similarity via pgvector, fused in a single SQL query. No separate BM25 process
to keep in sync. We'll build exactly that in Section 6.


---
## Section 3 — Ingest the corpus

Same 5 runbooks, same cleaning, same chunking as Notebooks 1–2 — now with metadata attached and
written transactionally.


In [ ]:
RAW_RUNBOOKS = {
    "payment-service": """
CONFIDENTIAL - ACME Corp Internal
Page 1 of 2

RUNBOOK: Payment Service Outage

Symptoms:
- 5xx errors on /api/v1/charge endpoint
- Latency above 2000ms on payment-api
- Redis connection timeouts in payment logs

Investigation steps:
1. Check Kubernetes pods for the payment namespace.
   kubectl get pods -n payment
2. Inspect recent logs for exceptions.
   kubectl logs deployment/payment-api -n payment --tail=200
3. Verify Redis connectivity from a payment pod.
   kubectl exec -it payment-api-0 -- redis-cli -h redis-payment ping

Remediation:
If pods are in CrashLoopBackOff, restart the deployment:
   kubectl rollout restart deployment payment-api -n payment
If Redis is unreachable, failover to the replica:
   redis-cli -h redis-payment SENTINEL failover payment-master

Escalation: page the payments on-call if errors persist 15 minutes.

CONFIDENTIAL - ACME Corp Internal
Page 2 of 2
""",
    "order-service": """
CONFIDENTIAL - ACME Corp Internal
Page 1 of 1

RUNBOOK: Order Service Degradation

Symptoms:
- Orders stuck in PENDING state
- Growing queue depth on order-events topic
- Timeouts calling payment-service downstream

Investigation steps:
1. Check order-service pod health.
   kubectl get pods -n orders
2. Check the dead letter queue for poison messages.
   kubectl logs deployment/order-worker -n orders | grep DLQ
3. Verify downstream payment-service is healthy before restarting anything.

Remediation:
Restart the order worker to reprocess stuck orders:
   kubectl rollout restart deployment order-worker -n orders
Replay dead-lettered events after fixing the schema issue.

CONFIDENTIAL - ACME Corp Internal
""",
    "login-service": """
CONFIDENTIAL - ACME Corp Internal

RUNBOOK: Login Service Authentication Failures

Symptoms:
- Spike in 401 responses on /auth/login
- JWT signature validation errors in logs
- Session store (Redis) memory above 90 percent

Investigation steps:
1. Confirm the JWT signing key was not rotated without deployment.
   kubectl get secret jwt-signing-key -n auth -o yaml
2. Check Redis session store memory.
   redis-cli -h redis-sessions INFO memory
3. Review recent deployments to login-service.

Remediation:
Roll back the last deployment if key rotation caused the failure:
   kubectl rollout undo deployment login-service -n auth
Evict expired sessions if Redis memory is exhausted.

Page the identity team if MFA providers are timing out.

CONFIDENTIAL - ACME Corp Internal
""",
    "kafka-cluster": """
CONFIDENTIAL - ACME Corp Internal
Page 1 of 2

RUNBOOK: Kafka Cluster Incidents

Symptoms:
- Consumer lag increasing on critical topics
- Under-replicated partitions alert firing
- ISR shrinking on broker 2
- Producers receiving NotEnoughReplicasException

Investigation steps:
1. Check broker health and disk usage.
   kafka-broker-api-versions.sh --bootstrap-server kafka-0:9092
2. List under-replicated partitions.
   kafka-topics.sh --describe --under-replicated-partitions --bootstrap-server kafka-0:9092
3. Check consumer group lag.
   kafka-consumer-groups.sh --describe --group order-consumers --bootstrap-server kafka-0:9092

Remediation:
If a broker is down due to disk pressure, clear old log segments and restart the broker:
   systemctl restart kafka
If consumer lag keeps growing, scale the consumer group before touching the brokers.
Never delete topics during an incident.

CONFIDENTIAL - ACME Corp Internal
Page 2 of 2
""",
    "kubernetes-deploys": """
CONFIDENTIAL - ACME Corp Internal

RUNBOOK: Kubernetes Deployment Failures

Symptoms:
- Pods stuck in ImagePullBackOff or CrashLoopBackOff
- Rollout stuck at 50 percent
- Readiness probes failing after a new release

Investigation steps:
1. Describe the failing pod to see events.
   kubectl describe pod <pod-name>
2. Check rollout status.
   kubectl rollout status deployment <name>
3. Compare the new image tag against the registry.

Remediation:
Roll back a bad release immediately:
   kubectl rollout undo deployment <name>
Fix readiness probe thresholds if the app needs longer warmup.
Always roll back first, debug second, during a customer-facing incident.

CONFIDENTIAL - ACME Corp Internal
""",
}

# metadata we would really have in a runbook repo
DOC_META = {
    "payment-service":    {"service": "payment-service",    "severity": "critical"},
    "order-service":      {"service": "order-service",      "severity": "high"},
    "login-service":      {"service": "login-service",      "severity": "critical"},
    "kafka-cluster":      {"service": "kafka",              "severity": "critical"},
    "kubernetes-deploys": {"service": "platform",           "severity": "medium"},
}

def clean(text):
    for pat in [r"CONFIDENTIAL.*", r"Page \d+ of \d+"]:
        text = re.sub(pat, "", text)
    text = re.sub(r"[ \t]+", " ", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()

def chunk_words(text, chunk_size, overlap=0):
    words, step, out = text.split(), chunk_size - overlap, []
    for start in range(0, len(words), step):
        piece = words[start:start + chunk_size]
        if len(piece) < 5: break
        out.append(" ".join(piece))
        if start + chunk_size >= len(words): break
    return out

CHUNK_SIZE, OVERLAP = 60, 12
MODEL_NAME = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(MODEL_NAME)

import hashlib
rows = []
for doc, raw in RAW_RUNBOOKS.items():
    for i, ch in enumerate(chunk_words(clean(raw), CHUNK_SIZE, OVERLAP)):
        rows.append({
            "doc": doc, "chunk_id": f"{doc}#{i}", "chunk_index": i, "content": ch,
            "content_hash": hashlib.sha256(ch.encode()).hexdigest()[:16],
            "service": DOC_META[doc]["service"], "severity": DOC_META[doc]["severity"],
        })

embs = embedder.encode([r["content"] for r in rows],
                       normalize_embeddings=True, show_progress_bar=True).astype("float32")

with conn.cursor() as cur:
    execute_values(cur, """
        INSERT INTO runbook_chunks
          (doc, chunk_id, chunk_index, content, content_hash, service, severity, embedding, embedding_model)
        VALUES %s
    """, [(r["doc"], r["chunk_id"], r["chunk_index"], r["content"], r["content_hash"],
           r["service"], r["severity"], embs[i], MODEL_NAME) for i, r in enumerate(rows)])

with conn.cursor() as cur:
    cur.execute("SELECT count(*), count(DISTINCT doc) FROM runbook_chunks;")
    n, d = cur.fetchone()
print(f"\nIngested {n} chunks from {d} runbooks ✅")

### 3.1 Idempotent re-ingestion — the `content_hash` payoff

Run ingestion twice and you get duplicates, which quietly poison retrieval (the same chunk occupies
multiple top-k slots). Real pipelines are **idempotent**: `ON CONFLICT` makes re-running safe.


In [ ]:
with conn.cursor() as cur:
    # Simulate re-ingesting the same corpus: only genuinely changed chunks should be re-embedded
    cur.execute("SELECT chunk_id, content_hash FROM runbook_chunks;")
    existing = dict(cur.fetchall())

unchanged = sum(1 for r in rows if existing.get(r["chunk_id"]) == r["content_hash"])
print(f"{unchanged}/{len(rows)} chunks unchanged → skip re-embedding")
print(f"Embedding calls avoided: {unchanged}  (at scale this is the difference between $0.20 and $200)")

# The idempotent write pattern
UPSERT = """
INSERT INTO runbook_chunks
  (doc, chunk_id, chunk_index, content, content_hash, service, severity, embedding, embedding_model)
VALUES %s
ON CONFLICT (chunk_id) DO UPDATE SET
  content = EXCLUDED.content,
  content_hash = EXCLUDED.content_hash,
  embedding = EXCLUDED.embedding,
  embedding_model = EXCLUDED.embedding_model,
  updated_at = now()
WHERE runbook_chunks.content_hash IS DISTINCT FROM EXCLUDED.content_hash;
"""
with conn.cursor() as cur:
    execute_values(cur, UPSERT,
        [(r["doc"], r["chunk_id"], r["chunk_index"], r["content"], r["content_hash"],
          r["service"], r["severity"], embs[i], MODEL_NAME) for i, r in enumerate(rows)])
    cur.execute("SELECT count(*) FROM runbook_chunks;")
    print(f"\nRow count after re-running ingestion: {cur.fetchone()[0]}  (unchanged → idempotent ✅)")

---
## Section 4 — Vector search in SQL

pgvector adds distance operators. The one you use must match how you built your vectors:

| Operator | Distance | Use when |
|---|---|---|
| `<=>` | cosine | vectors normalized or not — the safe default for text |
| `<#>` | negative inner product | vectors are unit-normalized (fastest) |
| `<->` | L2 / Euclidean | rarely right for text embeddings |

`ORDER BY embedding <=> query` sorts **ascending by distance** — nearest first. Similarity is
`1 - distance`. Getting this backwards silently returns the *least* relevant chunks, and it's a
genuinely common bug because the query still "works".


In [ ]:
def embed(q):
    return embedder.encode([q], normalize_embeddings=True).astype("float32")[0]
    # [0] unwraps the batch dimension — encode() always returns a LIST of vectors (one per input
    # text), and we passed a list of one question, so we take the single vector back out

def vector_search(question, k=5, service=None, since_days=None):
    """Vector search with optional metadata pre-filtering — all in one SQL statement."""
    # Build the WHERE clause DYNAMICALLY based on which filters were requested. `where` collects
    # SQL fragments as strings; `params` collects the actual values, kept separate so psycopg2 can
    # safely parameterize them (never string-format a value directly into SQL — injection risk).
    where, params = [], {"q": embed(question), "k": k}
    if service:
        where.append("service = %(service)s"); params["service"] = service
    if since_days:
        where.append("updated_at > now() - make_interval(days => %(days)s)"); params["days"] = since_days
    clause = ("WHERE " + " AND ".join(where)) if where else ""   # join with AND, or skip WHERE entirely

    sql = f"""
        SELECT chunk_id, doc, service, content,
               1 - (embedding <=> %(q)s) AS similarity
        FROM runbook_chunks
        {clause}
        ORDER BY embedding <=> %(q)s
        LIMIT %(k)s;
    """
    # `<=>` is pgvector's COSINE DISTANCE operator: 0 = identical direction, 2 = opposite.
    # ORDER BY it ascending puts the CLOSEST (most similar) rows first — that's why we sort by
    # distance but then compute similarity = 1 - distance for the human-readable output column.
    with conn.cursor(cursor_factory=RealDictCursor) as cur:
        cur.execute(sql, params)     # %(q)s / %(k)s etc. are filled in safely from the params dict —
        return pd.DataFrame(cur.fetchall())   # this is psycopg2's parameterized query syntax

q = "The message bus is falling behind"
print(f'QUERY: "{q}"\n')
display(vector_search(q, k=5)[["similarity", "chunk_id", "service"]])

print("\nSame query, PRE-FILTERED to service='payment-service' (the SQL WHERE does the work):")
display(vector_search("How do I restart it?", k=3, service="payment-service")[["similarity", "chunk_id"]])

**Compare this to Notebook 2 §7.** There, metadata pre-filtering meant hand-slicing numpy arrays
and passing `candidate_ids` around. Here it's a `WHERE` clause the query planner optimizes, combined
with the B-tree index we created. This is the single most practical argument for pgvector: **filtering
and vector search live in the same engine.**


---
## Section 5 — HNSW: buying speed with recall

Our queries so far did a **sequential scan** — pgvector compared the query against *every* row.
Exact, but O(n).

**HNSW** (Hierarchical Navigable Small World) builds a layered proximity graph and walks it greedily.
Sub-linear search, but **approximate** — it can miss the true nearest neighbor.

Three parameters, and you should be able to explain all three:

| Parameter | When | Effect |
|---|---|---|
| `m` | build | edges per node. Higher → better recall, more memory, slower build |
| `ef_construction` | build | candidate list size while building. Higher → better graph, slower build |
| `ef_search` | **query** | candidates explored per search. **Higher → better recall, slower query** |

`ef_search` is the runtime dial — the one you tune in production without rebuilding.

**The honest caveat for this notebook:** we have ~10 chunks. HNSW on 10 rows is meaningless — Postgres
will often ignore the index entirely because a seq scan is cheaper. So we do the *measurement* on
synthetic vectors at a realistic scale, where the recall trade-off is actually visible.


In [ ]:
# Build HNSW on the real (tiny) table — correct DDL, even if the planner won't use it at n=10
with conn.cursor() as cur:
    cur.execute("""
        CREATE INDEX idx_chunks_hnsw ON runbook_chunks
        USING hnsw (embedding vector_cosine_ops)
        WITH (m = 16, ef_construction = 64);
    """)
    cur.execute("ANALYZE runbook_chunks;")
print("HNSW index created ✅\n")

with conn.cursor() as cur:
    cur.execute("EXPLAIN ANALYZE SELECT chunk_id FROM runbook_chunks "
                "ORDER BY embedding <=> %s LIMIT 3;", (embed("kafka lag"),))
    for line in cur.fetchall():
        print(line[0])
print("\n↑ At n=10 the planner may well choose a Seq Scan — that is CORRECT behaviour, not a bug.")

In [ ]:
# ---- Measure the recall/latency trade-off at realistic scale --------
N_SYNTH, DIM = 50_000, 384
N_CLUSTERS, NOISE = 200, 0.05     # ← these numbers matter enormously; see below

print(f"Generating {N_SYNTH:,} synthetic vectors...")
rng = np.random.default_rng(42)

# (A) NAIVE: pure random Gaussian vectors — the obvious choice, and WRONG.
rand_vecs = rng.normal(size=(N_SYNTH, DIM)).astype("float32")
rand_vecs /= np.linalg.norm(rand_vecs, axis=1, keepdims=True)

# (B) REALISTIC: topic centroids + small noise, mimicking how real embeddings cluster.
centroids = rng.normal(size=(N_CLUSTERS, DIM))
centroids /= np.linalg.norm(centroids, axis=1, keepdims=True)
assign = rng.integers(0, N_CLUSTERS, N_SYNTH)
synth = centroids[assign] + rng.normal(scale=NOISE, size=(N_SYNTH, DIM))
synth = (synth / np.linalg.norm(synth, axis=1, keepdims=True)).astype("float32")

# Prove the difference is real, before we index anything
def peak_similarity(mat, n=300):
    s = mat[:n] @ mat[:n].T
    np.fill_diagonal(s, -9)
    return s.max()

print(f"\n  (A) random vectors    : peak cosine similarity = {peak_similarity(rand_vecs):.3f}")
print(f"  (B) clustered vectors: peak cosine similarity = {peak_similarity(synth):.3f}")
print("\nReal sentence embeddings show peak intra-topic similarity around 0.5-0.9,")
print("so (B) is the honest analogue. Section 5.2 shows why this choice decides the whole result.")

with conn.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS synth_vectors;")
    cur.execute("CREATE TABLE synth_vectors (id BIGSERIAL PRIMARY KEY, embedding vector(384));")
    execute_values(cur, "INSERT INTO synth_vectors (embedding) VALUES %s",
                   [(v,) for v in synth], page_size=1000)
    cur.execute("SELECT count(*) FROM synth_vectors;")
    print(f"\nInserted {cur.fetchone()[0]:,} clustered vectors")

In [ ]:
# Queries must come from the SAME distribution as the corpus.
# A query drawn at random would sit in empty space, and recall would be meaningless.
QUERIES = []
for _ in range(20):
    c = centroids[rng.integers(0, N_CLUSTERS)] + rng.normal(scale=NOISE, size=DIM)
    QUERIES.append((c / np.linalg.norm(c)).astype("float32"))
K = 10

def topk_ids(qv, k=K, use_index=True, ef_search=None):
    """Run one top-k search and return (which row ids came back, how long it took).
    use_index toggles whether Postgres is ALLOWED to use the HNSW index at all — that's
    what lets us compute both the exact baseline and the approximate result with the same function."""
    with conn.cursor() as cur:
        # SET enable_seqscan controls the query PLANNER, not the data — 'off' means "don't even
        # consider a sequential scan", which forces Postgres to use the HNSW index if one exists.
        # 'on' (the default) lets the planner choose freely, which at small n often means seq scan
        # anyway (see the EXPLAIN ANALYZE output above) — so for the exact baseline we instead just
        # make sure NO HNSW index exists yet (see the DROP INDEX below), guaranteeing an exact scan.
        cur.execute("SET enable_seqscan = %s;", ("on" if not use_index else "off",))
        if ef_search is not None:
            cur.execute("SET hnsw.ef_search = %s;", (ef_search,))   # the ONE knob tunable per query,
                                                                      # no index rebuild required
        t0 = time.perf_counter()
        cur.execute("SELECT id FROM synth_vectors ORDER BY embedding <=> %s LIMIT %s;", (qv, k))
        ids = [r[0] for r in cur.fetchall()]
        return ids, (time.perf_counter() - t0) * 1000    # *1000 converts seconds to milliseconds

# --- GROUND TRUTH: exact search, no index -------------------------------
# recall is only measurable if we know the TRUE top-k first. Drop the HNSW index (if any exists
# from a previous run) so this query has NO CHOICE but to do an exact sequential scan.
with conn.cursor() as cur:
    cur.execute("DROP INDEX IF EXISTS idx_synth_hnsw;")
truth, exact_ms = {}, []
for i, qv in enumerate(QUERIES):
    ids, ms = topk_ids(qv, use_index=False)
    truth[i] = set(ids)          # store as a SET (not list) — makes the recall calculation below
    exact_ms.append(ms)          # a simple set-intersection instead of manual list comparison
print(f"Exact (seq scan) baseline: {np.mean(exact_ms):.1f} ms/query")

# --- Build HNSW ---------------------------------------------------------
print("\nBuilding HNSW index (m=16, ef_construction=64)...")
t0 = time.perf_counter()
with conn.cursor() as cur:
    cur.execute("CREATE INDEX idx_synth_hnsw ON synth_vectors "
                "USING hnsw (embedding vector_cosine_ops) WITH (m=16, ef_construction=64);")
    cur.execute("ANALYZE synth_vectors;")
print(f"Build took {time.perf_counter()-t0:.1f}s")

In [ ]:
results = []
for ef in [10, 20, 40, 80, 160, 320]:            # sweep the ONE tunable knob across a wide range
    recalls, times = [], []
    for i, qv in enumerate(QUERIES):
        ids, ms = topk_ids(qv, use_index=True, ef_search=ef)   # this time WITH the HNSW index forced on
        # recall@10 = (how many of the approximate top-10 are ALSO in the true top-10) / 10.
        # set(ids) & truth[i] is set intersection — the overlap between what HNSW returned and
        # what we know is actually correct from the exact-scan ground truth computed above.
        recalls.append(len(set(ids) & truth[i]) / K)
        times.append(ms)
    results.append({"ef_search": ef, "recall@10": np.mean(recalls),   # average over all 20 queries
                    "latency_ms": np.mean(times)})

df_hnsw = pd.DataFrame(results)
df_hnsw["speedup_vs_exact"] = np.mean(exact_ms) / df_hnsw.latency_ms   # how many times faster than
print(df_hnsw.to_string(index=False))                                  # the exact-scan baseline

# Two different y-axes sharing one x-axis: recall (0-1 scale) and latency (milliseconds) live on
# very different scales, so a single shared axis would squash one of them flat. twinx() gives the
# second line its own right-hand y-axis while keeping both plotted against the same ef_search x-axis.
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(df_hnsw.ef_search, df_hnsw["recall@10"], "o-", color="#4C72B0", label="recall@10")
ax1.set_xlabel("ef_search"); ax1.set_ylabel("recall@10", color="#4C72B0")
ax1.set_ylim(0, 1.05); ax1.set_xscale("log", base=2); ax1.grid(alpha=0.3)   # log2 scale since ef_search
                                                                              # values double each step
ax2 = ax1.twinx()                                    # second axes object sharing ax1's x-axis
ax2.plot(df_hnsw.ef_search, df_hnsw.latency_ms, "s--", color="#C44E52", label="latency")
ax2.set_ylabel("latency (ms)", color="#C44E52")
ax1.axhline(np.mean(exact_ms), color="gray", ls=":", alpha=0)
plt.title(f"HNSW: recall vs latency ({N_SYNTH:,} vectors)  —  exact scan = {np.mean(exact_ms):.0f} ms")
plt.tight_layout(); plt.show()

### 5.2 The benchmark trap — why the *data* decided this result

Everything above depends on a choice buried in one line: `NOISE = 0.05`.

While building this notebook I first generated the synthetic corpus as **pure random Gaussian
vectors** — the obvious thing to do. Here are the real measured results from that version, on
identical hardware, index parameters, and query count:

| ef_search | recall@10 (random) | recall@10 (clustered) |
|---|---|---|
| 10 | 0.025 | 0.91 |
| 20 | 0.065 | 0.99 |
| 40 | 0.085 | 1.00 |
| 80 | 0.155 | 1.00 |
| 160 | 0.300 | 1.00 |

Same index. Same code. **Recall of 0.03 versus 0.91.** A reader running the random version would
reasonably conclude that HNSW is unusable and abandon it.

**Why random vectors break HNSW.** In 384 dimensions, independent Gaussian vectors are all nearly
orthogonal to each other — the *curse of dimensionality*. Measured peak cosine similarity across
random vectors is ~0.23, meaning no vector is meaningfully closer to any other. HNSW works by
building a navigable graph out of neighbourhood structure; when there is no structure, the greedy
walk has nothing to follow and degenerates. Real embeddings are the opposite: sentences about Kafka
cluster tightly together, which is precisely the structure HNSW exploits.

**The transferable lesson, and it is a big one:** *a benchmark on unrealistic data doesn't produce a
slightly-wrong answer — it produces a confidently inverted one.* Before trusting any ANN benchmark
(including vendor ones), check whether the test vectors have the clustering properties of your real
corpus. When you tune `ef_search` for the SRE Copilot in Milestone 13, run the sweep against
**embeddings of your actual runbooks**, never synthetic stand-ins.

*(You can verify this yourself: change `NOISE` to `0.35` in §5.1, re-run, and watch recall collapse.)*


**Reading the chart above.** Read it as an SRE reads a latency/cost curve:

- Recall climbs with `ef_search` and **plateaus** — past that point you pay latency for nothing.
- The exact scan is the recall=1.0 reference. HNSW's whole value is landing near it, far faster.
- **Recall < 1.0 means your RAG silently loses documents.** No error, no log line — the right chunk
  just isn't in the candidate set, and the LLM answers from whatever was.

**Production practice:** pick `ef_search` by measuring recall on *your* data, not by copying a default.
Then re-measure after significant corpus growth — the graph's structure changes as you add vectors.
This is exactly the kind of measurement Milestone 13 should produce as an artifact.


---
## Section 6 — Hybrid search, entirely in SQL

Notebook 2 ran BM25 in a separate Python process and fused in numpy. Two systems, two places to drift.

Postgres does full-text search natively (`ts_rank` over the GIN index we built in §2), so we can fuse
keyword and vector ranking in **one query** using Reciprocal Rank Fusion — implemented with CTEs and
window functions.


In [ ]:
HYBRID_SQL = """
WITH vector_ranked AS (
    SELECT id, chunk_id, doc, service, content,
           ROW_NUMBER() OVER (ORDER BY embedding <=> %(q)s) AS vrank
    FROM runbook_chunks
    {where_v}
    ORDER BY embedding <=> %(q)s
    LIMIT %(cand)s
),
keyword_ranked AS (
    SELECT id, chunk_id, doc, service, content,
           ROW_NUMBER() OVER (
               ORDER BY ts_rank(to_tsvector('english', content),
                                plainto_tsquery('english', %(qtext)s)) DESC
           ) AS krank
    FROM runbook_chunks
    WHERE to_tsvector('english', content) @@ plainto_tsquery('english', %(qtext)s)
    {and_k}
    LIMIT %(cand)s
)
SELECT
    COALESCE(v.chunk_id, k.chunk_id)  AS chunk_id,
    COALESCE(v.doc, k.doc)            AS doc,
    COALESCE(v.content, k.content)    AS content,
    v.vrank                           AS vector_rank,
    k.krank                           AS keyword_rank,
    -- 60 is the standard RRF smoothing constant. Cast to numeric so this is
    -- NOT integer division (a classic silent bug: 1/(60+rank) would floor to 0).
    COALESCE(1.0::numeric / (60 + v.vrank), 0) +
    COALESCE(1.0::numeric / (60 + k.krank), 0) AS rrf_score
FROM vector_ranked v
FULL OUTER JOIN keyword_ranked k USING (id)
ORDER BY rrf_score DESC
LIMIT %(k)s;
"""

def hybrid_search(question, k=5, candidates=20, service=None):
    where_v = "WHERE service = %(service)s" if service else ""
    and_k   = "AND service = %(service)s"   if service else ""
    sql = HYBRID_SQL.format(where_v=where_v, and_k=and_k)
    params = {"q": embed(question), "qtext": question, "k": k, "cand": candidates}
    if service: params["service"] = service
    with conn.cursor(cursor_factory=RealDictCursor) as cur:
        cur.execute(sql, params)
        out = pd.DataFrame(cur.fetchall())
    # Postgres NUMERIC arrives as Decimal; force float so downstream .max()/
    # comparisons behave. Also guarantee columns exist even on zero rows.
    if out.empty:
        return pd.DataFrame(columns=["chunk_id", "doc", "content",
                                     "vector_rank", "keyword_rank", "rrf_score"])
    out["rrf_score"] = out["rrf_score"].astype(float)
    return out

for q in ["NotEnoughReplicasException",          # exact identifier → keyword should win
          "The message bus is falling behind"]:  # paraphrase → vector should win
    print(f'QUERY: "{q}"')
    r = hybrid_search(q, k=4)
    display(r[["chunk_id", "vector_rank", "keyword_rank", "rrf_score"]])

**Read the `vector_rank` / `keyword_rank` columns side by side** — they show each retriever's
independent opinion. A `NULL` means that retriever didn't return the chunk at all. On the exact
identifier query, keyword search should rank the Kafka chunk highly while the vector side is vaguer;
on the paraphrase, the reverse. RRF fuses them without either dominating.

**Why RRF over weighted score blending here:** `ts_rank` and cosine similarity are on completely
different scales, and `ts_rank` values shift as the corpus grows. Rank-based fusion is immune to both.
That's why it's the default in production hybrid systems — one less thing to re-tune.


---
## Section 7 — Evaluation, implemented from scratch

`hit@1` was a fine starter metric, but it only asks *"was the top result right?"* Real evaluation asks
richer questions. Here are the four standard IR metrics — each is a few lines of arithmetic.

| Metric | Question it answers |
|---|---|
| **Precision@k** | Of the k I returned, what fraction were relevant? → *prompt pollution* |
| **Recall@k** | Of all relevant chunks, what fraction did I find? → *missed evidence* |
| **MRR** | How high was the *first* relevant result? → *does the user see it immediately* |
| **nDCG@k** | Are relevant results ranked near the top? (position-weighted) → *overall ranking quality* |

**Which one matters for the SRE Copilot?** Precision, mostly. A wrong chunk in the prompt is an
invitation to hallucinate a remediation step, and remediation steps get *executed* during incidents.
Recall matters too — but for a 3-chunk prompt, precision is the constraint that binds.


In [ ]:
def precision_at_k(rel, k):
    # rel is a 0/1 array: rel[i]=1 means the chunk at rank i is actually relevant. Precision@k asks:
    # of the top k results we RETURNED, what fraction were relevant? mean() of 0/1s = fraction of 1s.
    return float(np.mean(rel[:k])) if k else 0.0

def recall_at_k(rel, k, total_relevant):
    # Of ALL the relevant chunks that exist in the corpus (total_relevant), how many did we
    # find within our top k? sum(rel[:k]) counts relevant hits found; divide by the true total.
    return float(np.sum(rel[:k]) / total_relevant) if total_relevant else 0.0

def reciprocal_rank(rel):
    # Where does the FIRST relevant result appear? np.flatnonzero(rel) returns the indices where
    # rel is nonzero (i.e. relevant), in order — idx[0] is the position of the first one, 0-indexed.
    # +1 converts to 1-indexed rank, then we take the reciprocal: rank 1 -> score 1.0, rank 2 -> 0.5,
    # rank 10 -> 0.1. A steep penalty for making the user scroll past irrelevant results.
    idx = np.flatnonzero(rel)
    return float(1.0 / (idx[0] + 1)) if len(idx) else 0.0    # empty (no relevant results at all) -> 0

def dcg_at_k(rel, k):
    # Discounted Cumulative Gain: relevant results are worth less the further down they appear.
    # np.log2(np.arange(2, len(rel)+2)) generates the discount denominators log2(2), log2(3), log2(4)...
    # for ranks 1, 2, 3... — rank 1 gets divided by log2(2)=1 (no discount), rank 2 by log2(3)≈1.58
    # (discounted), and so on. Sum of (relevance / discount) across the top k results.
    rel = np.asarray(rel[:k], dtype=float)
    return float(np.sum(rel / np.log2(np.arange(2, len(rel) + 2))))

def ndcg_at_k(rel, k):
    # NORMALIZED DCG: raw DCG isn't comparable across queries with different numbers of relevant
    # docs, so we divide by the IDEAL DCG — what DCG WOULD be if every relevant result were ranked
    # first (np.sort(rel)[::-1] sorts descending, pushing all 1s to the front — the best possible order).
    # This normalizes every query's score to the same 0-1 scale: 1.0 = perfect ranking.
    ideal = dcg_at_k(np.sort(rel)[::-1], k)
    return float(dcg_at_k(rel, k) / ideal) if ideal > 0 else 0.0    # ideal=0 means no relevant docs exist at all

# --- sanity checks: never trust a metric you haven't tested -------------
perfect = np.array([1, 1, 1, 0, 0, 0])    # all 3 relevant docs ranked first — best possible ordering
worst   = np.array([0, 0, 0, 1, 1, 1])    # all 3 relevant docs ranked LAST — worst possible ordering
mixed   = np.array([0, 1, 0, 1, 0, 0])    # relevant docs at positions 2 and 4 (1-indexed)
print(f"nDCG@3 perfect ranking = {ndcg_at_k(perfect,3):.3f}   (expect 1.000)")
print(f"nDCG@3 worst ranking   = {ndcg_at_k(worst,3):.3f}   (expect 0.000)")
print(f"MRR    first hit @2    = {reciprocal_rank(mixed):.3f}   (expect 0.500)")
print(f"P@3    mixed           = {precision_at_k(mixed,3):.3f}")
print(f"R@3    mixed (2 rel)   = {recall_at_k(mixed,3,2):.3f}")

### 7.1 A labelled evaluation set

Ground truth is **relevance labels**, not just the right document. Here we label at document level
(any chunk from the expected runbook counts as relevant) — cheap and reasonable. Chunk-level labels
are more precise but far more expensive to produce; that trade-off is itself a real project decision.


In [ ]:
EVAL_SET = [
    # (question, relevant_doc, category)
    ("How do I restart the payment service?",        "payment-service",    "lexical"),
    ("Why is Kafka consumer lag increasing?",        "kafka-cluster",      "lexical"),
    ("Users are getting 401 errors on login",        "login-service",      "lexical"),
    ("Orders are stuck in pending state",            "order-service",      "lexical"),
    ("How do I fix under-replicated partitions?",    "kafka-cluster",      "lexical"),
    ("The message bus is falling behind",            "kafka-cluster",      "semantic"),
    ("Sign-in is broken for everyone",               "login-service",      "semantic"),
    ("Customers see failures at checkout",           "payment-service",    "semantic"),
    ("The event pipeline stopped delivering data",   "kafka-cluster",      "semantic"),
    ("Purchases are timing out",                     "order-service",      "semantic"),
    ("NotEnoughReplicasException in producer logs",  "kafka-cluster",      "identifier"),
    ("payment-api-0 pod is unhealthy",               "payment-service",    "identifier"),
    ("ImagePullBackOff on new release",              "kubernetes-deploys", "identifier"),
]

with conn.cursor() as cur:
    cur.execute("SELECT doc, count(*) FROM runbook_chunks GROUP BY doc;")
    CHUNKS_PER_DOC = dict(cur.fetchall())

def evaluate_retriever(retrieve_fn, name, k=5):
    """retrieve_fn(question, k) -> DataFrame with a 'doc' column, ranked best-first."""
    per_q = []
    for question, gold_doc, category in EVAL_SET:
        res = retrieve_fn(question, k)
        docs = res["doc"].tolist() if not res.empty else []
        rel = np.array([1 if d == gold_doc else 0 for d in docs])
        total_rel = CHUNKS_PER_DOC.get(gold_doc, 1)
        per_q.append({
            "question": question, "category": category, "gold": gold_doc,
            "P@3": precision_at_k(rel, 3), "P@5": precision_at_k(rel, 5),
            "R@5": recall_at_k(rel, 5, total_rel),
            "MRR": reciprocal_rank(rel), "nDCG@5": ndcg_at_k(rel, 5),
            "hit@1": float(rel[0]) if len(rel) else 0.0,
        })
    d = pd.DataFrame(per_q)
    summary = d[["P@3", "P@5", "R@5", "MRR", "nDCG@5", "hit@1"]].mean().to_dict()
    summary["retriever"] = name
    return d, summary

def vec_fn(q, k):    return vector_search(q, k=k)
def hyb_fn(q, k):    return hybrid_search(q, k=k)

det_vec, sum_vec = evaluate_retriever(vec_fn, "vector only")
det_hyb, sum_hyb = evaluate_retriever(hyb_fn, "hybrid (RRF)")

board = pd.DataFrame([sum_vec, sum_hyb]).set_index("retriever").round(3)
display(board)

In [ ]:
# Where does each retriever win or lose? Break down by question category.
det_vec["retriever"] = "vector"; det_hyb["retriever"] = "hybrid"
both = pd.concat([det_vec, det_hyb])
pivot = both.pivot_table(index="category", columns="retriever", values="nDCG@5", aggfunc="mean").round(3)
display(pivot)

pivot.plot(kind="bar", figsize=(9, 5), color=["#4C72B0", "#55A868"])
plt.ylabel("nDCG@5"); plt.ylim(0, 1.05); plt.xticks(rotation=0)
plt.title("Retrieval quality by question type — where each approach earns its keep")
plt.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()

print("\nPer-question detail (hybrid):")
display(det_hyb[["category", "question", "hit@1", "P@3", "MRR", "nDCG@5"]])

> ⚠️ **Sample size, again.** 13 questions over ~10 chunks. A single question flipping moves a mean
> by ~0.08. Treat these numbers as an illustration of the *method*. The transferable practice is:
> label a set, break results down by failure category, and compare retrievers on identical inputs.
> When you build the real eval set for the SRE Copilot, harvest questions from actual Slack incident
> threads — that's the only way the distribution matches production.


---
## Section 8 — Observability

You instrument every other service in the SRE Copilot. Retrieval deserves the same treatment —
and it fails in ways that are *invisible* without instrumentation, because a bad retrieval still
returns a confident-looking answer.

**Four signals worth emitting:**
1. **Latency percentiles** (p50/p95/p99) per stage — embed, search, re-rank. Averages hide the tail.
2. **Top relevance score** per query — the drift alarm from Notebook 2 §8.1.
3. **Result count after thresholding** — a rise in zero-result queries means corpus/traffic divergence.
4. **Structured query logs** — so you can replay a bad incident answer and see *what was retrieved*.

That last one matters most. When an engineer says "the copilot gave me a wrong remediation," the first
question is: *was retrieval wrong, or was the LLM wrong given correct context?* Without a query log
you cannot answer that, and you'll waste the postmortem arguing about it.


In [ ]:
QUERY_LOG = []

def instrumented_search(question, k=3, service=None, min_score=None):
    """Retrieval with per-stage timing and structured logging."""
    t_start = time.perf_counter()

    t0 = time.perf_counter()
    qv = embed(question)
    t_embed = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    res = hybrid_search(question, k=k, service=service)
    t_search = (time.perf_counter() - t0) * 1000

    top_score = float(res["rrf_score"].max()) if not res.empty else 0.0
    if min_score is not None and not res.empty:
        res = res[res["rrf_score"] >= min_score]

    record = {
        "ts": pd.Timestamp.utcnow().isoformat(),
        "question": question,
        "service_filter": service,
        "n_results": int(len(res)),
        "top_score": round(top_score, 4),
        "embed_ms": round(t_embed, 2),
        "search_ms": round(t_search, 2),
        "total_ms": round((time.perf_counter() - t_start) * 1000, 2),
        "embedding_model": MODEL_NAME,
        "refused": bool(res.empty),
    }
    QUERY_LOG.append(record)
    return res, record

res, rec = instrumented_search("Why is Kafka consumer lag increasing?", k=3)
print("Structured log line (this is what ships to your log aggregator):")
print(json.dumps(rec, indent=2))

In [ ]:
# Simulate production traffic: a realistic mix, including out-of-scope questions
TRAFFIC = ([q for q, _, _ in EVAL_SET] * 3 + [
    "How do I rotate the TLS certificate on the mail server?",
    "What is the refund policy for enterprise customers?",
    "How do I bake sourdough bread?",
] * 2)

QUERY_LOG.clear()
for q in TRAFFIC:
    instrumented_search(q, k=3)

log_df = pd.DataFrame(QUERY_LOG)
print(f"Logged {len(log_df)} queries\n")

lat = log_df[["embed_ms", "search_ms", "total_ms"]].describe(percentiles=[.5, .95, .99])
display(lat.loc[["mean", "50%", "95%", "99%", "max"]].round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].hist(log_df.total_ms, bins=25, color="#4C72B0", edgecolor="white")
for p, c in [(50, "#55A868"), (95, "#DD8452"), (99, "#C44E52")]:
    v = np.percentile(log_df.total_ms, p)
    axes[0].axvline(v, color=c, ls="--", lw=2, label=f"p{p} = {v:.0f} ms")
axes[0].set_xlabel("total latency (ms)"); axes[0].set_ylabel("queries")
axes[0].set_title("Latency distribution — always look at the tail"); axes[0].legend()

known = log_df[~log_df.question.isin(["How do I rotate the TLS certificate on the mail server?",
                                      "What is the refund policy for enterprise customers?",
                                      "How do I bake sourdough bread?"])]
unknown = log_df[log_df.question.isin(["How do I rotate the TLS certificate on the mail server?",
                                       "What is the refund policy for enterprise customers?",
                                       "How do I bake sourdough bread?"])]
axes[1].hist(known.top_score, bins=20, alpha=0.75, label="in-scope", color="#55A868")
axes[1].hist(unknown.top_score, bins=20, alpha=0.75, label="out-of-scope", color="#C44E52")
axes[1].set_xlabel("top relevance score"); axes[1].set_title("Score distribution — your drift alarm")
axes[1].legend()
plt.tight_layout(); plt.show()

**The right-hand chart is your production alarm.** If in-scope and out-of-scope scores separate
cleanly, a threshold works. If they overlap, no constant threshold can distinguish them — which is
precisely the finding that justifies Milestone 20 (Confidence Score) as dedicated work.

**What to alert on, concretely:**

| Signal | Alert condition | What it means |
|---|---|---|
| p95 retrieval latency | > SLO for 5 min | index bloat, missing index, or DB contention |
| zero-result rate | rises sharply | corpus and traffic have diverged |
| mean top score | drifts down week over week | new questions your runbooks don't cover |
| `embedding_model` distinct values | > 1 | **a partial re-index — the mismatched-model bug, caught automatically** |

That last row is worth pausing on. Because we stamped every row with its model in §2, a one-line query
detects the failure that Notebook 1's quiz asked about — the one that produces no error and garbage results.


In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT embedding_model, count(*) AS chunks,
               min(updated_at) AS oldest, max(updated_at) AS newest
        FROM runbook_chunks GROUP BY embedding_model;
    """)
    consistency = pd.DataFrame(cur.fetchall(),
                               columns=["embedding_model", "chunks", "oldest", "newest"])
display(consistency)

if len(consistency) > 1:
    print("🚨 ALERT: multiple embedding models in one table — vectors are not comparable!")
else:
    print("✅ Single embedding model across the corpus — vector space is consistent.")

---
## Section 9 — Putting it together: the production retrieval function

Everything from Notebooks 1–3, in the shape it should have when Milestone 16 calls it as a LangGraph tool.


In [ ]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def retrieve(question, k=3, service=None, min_score=None,
             candidates=20, use_rerank=True):
    """Production retrieval: filter → hybrid → rerank → threshold → log.

    Returns (results_df, log_record). Empty df means 'no confident answer'.
    """
    t_start = time.perf_counter()
    stage1 = hybrid_search(question, k=candidates, service=service)
    t_retrieve = (time.perf_counter() - t_start) * 1000

    t0 = time.perf_counter()
    if use_rerank and len(stage1) > 1:
        stage1 = stage1.assign(score=reranker.predict(
            [[question, c] for c in stage1["content"]]))
    else:
        stage1 = stage1.assign(score=stage1["rrf_score"])
    t_rerank = (time.perf_counter() - t0) * 1000

    out = stage1.sort_values("score", ascending=False).head(k).reset_index(drop=True)
    top = float(out["score"].max()) if not out.empty else 0.0
    if min_score is not None:
        out = out[out["score"] >= min_score]

    record = {"question": question, "service_filter": service,
              "retrieve_ms": round(t_retrieve, 1), "rerank_ms": round(t_rerank, 1),
              "total_ms": round((time.perf_counter() - t_start) * 1000, 1),
              "top_score": round(top, 4), "n_results": len(out),
              "refused": bool(out.empty), "embedding_model": MODEL_NAME}
    return out, record


def answer(question, k=3, service=None, min_score=None):
    hits, log = retrieve(question, k=k, service=service, min_score=min_score)
    if hits.empty:
        return "I don't have a runbook for that.", hits, log
    ctx = "\n\n".join(f"[Source: {r.chunk_id} | relevance={r.score:.2f}]\n{r.content}"
                       for _, r in hits.iterrows())
    prompt = (f"You are an SRE assistant. Answer using ONLY the context below, citing chunk ids.\n"
              f"If the context does not answer the question, say you don't have a runbook.\n\n"
              f"=== CONTEXT ===\n{ctx}\n\n=== QUESTION ===\n{question}\n\n=== ANSWER ===")
    return prompt, hits, log


for q, svc in [("The message bus is falling behind", None),
               ("How do I restart it?", "payment-service"),
               ("How do I bake sourdough bread?", None)]:
    _, hits, log = answer(q, k=3, service=svc)
    print(f"Q: {q}   filter={svc}")
    print(f"   top_score={log['top_score']:+.3f}  n={log['n_results']}  "
          f"retrieve={log['retrieve_ms']}ms  rerank={log['rerank_ms']}ms")
    print(f"   → {hits.chunk_id.tolist() if not hits.empty else 'NO RESULTS'}\n")

**Notice what `retrieve()` returns:** results *and* a log record. That shape is deliberate — the
LangGraph node in Notebook 4 will write the log record into graph state, so the agent's trace contains
the evidence for every retrieval decision. During an incident postmortem, that trace is the difference
between "the AI was wrong" and "retrieval returned the wrong chunk at 14:32, here's the score."


---
## Conclusion & Quiz

**What you built:** a durable, filterable, observable retrieval service — pgvector storage with real
schema decisions, a measured HNSW recall/latency trade-off, hybrid search in one SQL query,
IR metrics you implemented yourself, and instrumentation that catches silent failures.

**Mapping to the main project:**

| Notebook 3 section | SRE Copilot milestone |
|---|---|
| §2 schema, §3 ingestion | Milestone 13 (pgvector) |
| §5 HNSW tuning | Milestone 13 + Milestone 37 (Scaling) |
| §7 evaluation | Milestone 32 (Agent Metrics) |
| §8 observability | Milestone 30 (OpenTelemetry), Milestone 31 (Grafana) |
| §9 `retrieve()` | Milestone 16 (Retrieval Tools) |

### Quiz — answer before Notebook 4

1. `embedding vector(384)` fixes dimensionality at DDL time. Write the migration plan for moving to a
   1024-dim model **with zero downtime**. What does the `embedding_model` column buy you during that migration?
2. Your HNSW recall@10 measures 0.92. Explain, in terms an on-call engineer would accept, what the
   missing 8% actually *does* to an incident investigation — and why no error appears in any log.
3. We used RRF instead of weighted score blending in §6. Give two concrete reasons why blending
   `ts_rank` with cosine similarity is fragile, referencing how each score behaves as the corpus grows.
4. For the SRE Copilot specifically, argue for **precision@3 over recall@10** as the headline retrieval
   metric — then give one scenario where that choice is wrong and recall matters more.
5. In §8 we alert when `embedding_model` has more than one distinct value. Describe the exact sequence
   of events in a deploy that would trigger this, and why the system would otherwise appear healthy.
6. **Design question:** a runbook is edited during an active incident. Walk through what must happen
   between the git commit and the next retrieval returning the updated text. Where would you put the
   consistency boundary, and what would you deliberately *not* guarantee?

**Next — Notebook 4: Agentic RAG.** LangGraph nodes and state, retrieval as a *tool* the agent chooses
to call, multi-step reasoning over an Incident Bundle, memory across investigations, and
human-in-the-loop approval. That's where these three notebooks converge with Milestones 14–20 of the
main project — and where we finally let LangChain in, on our own terms.
